# 04 - Analyse des resultats et des cas d'echec

In [1]:
import json
import sys
from collections import defaultdict
from pathlib import Path

_c = Path.cwd()
while _c != _c.parent and not (_c / "data" / "raw").is_dir():
    _c = _c.parent
import os
os.chdir(_c)
sys.path.insert(0, str(_c))

In [2]:
res = json.loads(
    Path("data/evaluation/retrieval_results.json").read_text(encoding="utf-8")
)
print("Agregats :", res["aggregates"])

Agregats : {'precision_at_1': 0.2, 'precision_at_5': 0.10400000000000001, 'recall_at_5': 0.018136077717000567, 'source_hit_at_1': 0.2, 'source_hit_at_5': 0.41333333333333333, 'mrr': 0.29333333333333333}


In [3]:
qs = json.loads(
    Path("data/evaluation/test_questions.json").read_text(encoding="utf-8")
)["questions"]
qmap = {q["id"]: q for q in qs}
agg = defaultdict(lambda: defaultdict(list))
for r in res["per_question"]:
    c = qmap[r["id"]].get("category", "?")
    agg[c]["p1"].append(r["precision_at_1"])
    agg[c]["h5"].append(r["source_hit_at_5"])
    agg[c]["mrr"].append(r["mrr"])
for c in sorted(agg):
    a = agg[c]
    n = len(a["p1"])
    print(
        f"{c}: n={n} P1={sum(a['p1']) / n:.3f} hit5={sum(a['h5']) / n:.3f} MRR={sum(a['mrr']) / n:.3f}"
    )

beneficiaires: n=6 P1=0.000 hit5=0.667 MRR=0.278
comparaison: n=5 P1=0.400 hit5=0.600 MRR=0.467
complexe: n=5 P1=0.000 hit5=0.200 MRR=0.100
cotisations: n=8 P1=0.000 hit5=0.375 MRR=0.125
definition: n=8 P1=0.375 hit5=0.750 MRR=0.562
exclusions: n=8 P1=0.375 hit5=0.375 MRR=0.375
fiscalite: n=5 P1=0.000 hit5=0.000 MRR=0.000
garanties: n=10 P1=0.000 hit5=0.100 MRR=0.050
hors_perimetre: n=7 P1=0.000 hit5=0.000 MRR=0.000
resiliation: n=5 P1=0.200 hit5=0.600 MRR=0.400
souscription: n=8 P1=0.750 hit5=0.875 MRR=0.812


In [4]:
print(
    "== 9 cas d echec du perimetre (source attendue absente des 5 premiers passages) =="
)
for r in res["per_question"]:
    q = qmap.get(r["id"], {})
    if q.get("category") != "hors_perimetre" and r["source_hit_at_5"] == 0:
        print(
            r["id"],
            q.get("category"),
            "-> attendu",
            r["expected_source"],
            "::",
            q.get("question"),
        )

== 9 cas d echec du perimetre (source attendue absente des 5 premiers passages) ==
Q004 definition -> attendu spec_produits.md :: Qu'est-ce qu'Horizon Retraite ?
Q007 definition -> attendu conditions_generales.md :: Quelles sont les garanties de Visa Études ?
Q009 garanties -> attendu conditions_generales.md :: Que se passe-t-il si l'adhérent de Visa Études décède avant l'échéance ?
Q010 garanties -> attendu conditions_generales.md :: Que couvre la garantie invalidité de Visa Études ?
Q011 garanties -> attendu conditions_generales.md :: Que se passe-t-il si l'enfant bénéficiaire décède ?
Q012 garanties -> attendu conditions_generales.md :: Qu'est-ce que la rente éducation ?
Q013 garanties -> attendu conditions_generales.md :: En quoi la garantie décès de Visa Études Plus est-elle renforcée ?
Q014 garanties -> attendu conditions_generales.md :: Qu'est-ce que la garantie scolaire de Visa Études Plus ?
Q015 garanties -> attendu conditions_generales.md :: L'invalidité partielle permanente 

In [5]:
if Path("data/evaluation/chunking_experiment.json").exists():
    ch = json.loads(
        Path("data/evaluation/chunking_experiment.json").read_text(encoding="utf-8")
    )
    for cfg, v in ch.items():
        print(cfg, "->", v)

c300_o30 -> {'n_chunks': 150, 'aggregates': {'precision_at_1': 0.25333333333333335, 'precision_at_5': 0.22133333333333335, 'recall_at_5': 0.03474055927543223, 'source_hit_at_1': 0.25333333333333335, 'source_hit_at_5': 0.7866666666666666, 'mrr': 0.43377777777777776}}
c300_o60 -> {'n_chunks': 155, 'aggregates': {'precision_at_1': 0.25333333333333335, 'precision_at_5': 0.2186666666666667, 'recall_at_5': 0.03480710983580424, 'source_hit_at_1': 0.25333333333333335, 'source_hit_at_5': 0.7733333333333333, 'mrr': 0.4242222222222222}}
c500_o50 -> {'n_chunks': 92, 'aggregates': {'precision_at_1': 0.21333333333333335, 'precision_at_5': 0.192, 'recall_at_5': 0.05579173376274825, 'source_hit_at_1': 0.21333333333333335, 'source_hit_at_5': 0.6266666666666667, 'mrr': 0.37577777777777777}}
c500_o100 -> {'n_chunks': 94, 'aggregates': {'precision_at_1': 0.26666666666666666, 'precision_at_5': 0.17866666666666667, 'recall_at_5': 0.049545893719806756, 'source_hit_at_1': 0.26666666666666666, 'source_hit_at_5